[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/example3_repressilator.ipynb)

# Example 3 - Oscillations, and why the network needs Fourier features

The first two notebooks lived at steady states. This one moves. The **repressilator** is
three genes wired in a repression cycle - `x3` represses `x1`, `x1` represses `x2`, `x2`
represses `x3` - and for the right parameters it never settles: it **oscillates**, the
canonical synthetic genetic clock.

Two lessons here, both from the paper:

1. Least squares can actually be *more accurate* on the parameters for this circuit, and yet
   at low-to-moderate noise **both methods keep the correct region**. An imperfect numerical
   fit can still be the right qualitative regime - the other side of the Example 2 coin.
2. Fitting an oscillation with a plain neural network is hard. Networks have a *spectral
   bias*: they learn smooth, low-frequency shapes first and struggle with periodic, higher-
   frequency signals. The fix is **Fourier feature mapping** - feed the network sines and
   cosines of time so periodicity is built in.

We sample from DSGRN parameter node **13**. (As before, each notebook stands alone:
*region recovery* means the learned parameters satisfy the same DSGRN inequalities as the
truth.)

In [ ]:
# Colab already has numpy/scipy/matplotlib/torch, so this cell is a no-op there.
# Uncomment if you are running somewhere that is missing a package.
# !pip -q install numpy scipy matplotlib torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import torch, torch.nn as nn
torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=3, suppress=True)

### The smooth switches

Production is again built from Hill functions. Every edge in the repressilator is a
repression, so only the repressing form `f-` (high below threshold, low above) appears.

In [ ]:
# --- Hill production functions: smooth switches (gamma normalized to 1) ---
def hill_act(x, L, U, th, d):   # activating: low L -> high U as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * xd / (td + xd)

def hill_rep(x, L, U, th, d):   # repressing: high U -> low L as x rises
    xd = np.power(x, d); td = np.power(th, d)
    return L + (U - L) * td / (td + xd)

In [ ]:
# --- the same two functions in torch (used inside the network's physics loss) ---
def hill_act_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * xd / (td + xd)

def hill_rep_t(x, L, U, th, d):
    xd = x.clamp(min=1e-9).pow(d); td = th.pow(d)
    return L + (U - L) * td / (td + xd)

## 1. The model and its DSGRN region (node 13)

$$\dot{x}_1=-x_1+f^-(x_3),\quad \dot{x}_2=-x_2+f^-(x_1),\quad \dot{x}_3=-x_3+f^-(x_2).$$

The region for an oscillating node has a clean reading: each variable's production range
`(L, U)` must straddle the threshold at which it represses the next gene, i.e.
`L < theta < U` around the cycle. If a variable's range did not cross the next threshold, it
could never switch its target off and on, and the oscillation would die. `in_region` below
is those three sandwich inequalities.

In [ ]:
P = dict(L31=1.0,U31=5.0,th12=3.0, L12=1.0,U12=5.0,th23=3.0,
         L23=1.0,U23=5.0,th31=3.0, d=10.0, g=1.0)

def rhs(t, x, p):
    x1, x2, x3 = x
    return [-p['g']*x1 + hill_rep(x3, p['L31'],p['U31'],p['th31'],p['d']),
            -p['g']*x2 + hill_rep(x1, p['L12'],p['U12'],p['th12'],p['d']),
            -p['g']*x3 + hill_rep(x2, p['L23'],p['U23'],p['th23'],p['d'])]

def in_region(p):   # DSGRN node 13: each threshold between its regulator's L and U
    return (p['L31'] < p['th12'] < p['U31'] and
            p['L12'] < p['th23'] < p['U12'] and
            p['L23'] < p['th31'] < p['U23'] and
            0 < p['L31'] < p['U31'] and 0 < p['L12'] < p['U12'] and
            0 < p['L23'] < p['U23'])

print('do the true parameters lie in node 13?', in_region(P))

In [ ]:
def simulate(rhs, p, x0, T, n):
    t = np.linspace(0.0, T, n)
    sol = solve_ivp(rhs, (0.0, T), x0, t_eval=t, args=(p,), rtol=1e-9, atol=1e-11)
    return t, sol.y.T   # times (n,), states (n, m)

def add_noise(y, ub_frac, scale, rng):
    yn = y + rng.uniform(-ub_frac*scale, ub_frac*scale, size=y.shape)
    return np.clip(yn, 0.0, None)   # concentrations stay non-negative

## 2. Simulate the oscillation and add noise

We integrate over a longer horizon so several periods are visible, then add noise. Watch a
single trajectory: the three genes take turns peaking, chasing each other around the cycle.

In [ ]:
T, n = 40.0, 200
x0s = [[1.,2.,3.],[2.,3.,1.],[3.,1.,2.],[1.,1.,4.]]
ts, xs = [], []
for x0 in x0s:
    t, y = simulate(rhs, P, x0, T, n); ts.append(t); xs.append(y)

gap = min(abs(P['L31']-P['U31']), abs(P['L12']-P['U12']), abs(P['L23']-P['U23']))
rng = np.random.default_rng(0)
xs_noisy = [add_noise(y, 0.25, gap, rng) for y in xs]

plt.figure(figsize=(8, 3.2))
plt.plot(ts[0], xs[0][:,0], label='$x_1$'); plt.plot(ts[0], xs[0][:,1], label='$x_2$')
plt.plot(ts[0], xs[0][:,2], label='$x_3$')
plt.plot(ts[0], xs_noisy[0][:,0], '.', ms=3, color='C0', alpha=.5)
plt.xlabel('t'); plt.legend(); plt.title('Repressilator (one trajectory, with noise)')
plt.show()

**What you should see.** Three interleaved oscillations, each gene peaking after the one
that represses it relaxes. The dots show how noise blurs the wave - and a smooth network
asked to fit that wave is exactly where spectral bias bites.

## 3. Least squares and the region check

Gradient-matching least squares again, over the nine parameters. For this circuit it does
well, and we confirm the recovered parameters keep node 13 across a noise sweep.

In [ ]:
KEYS = ['L31','U31','th31','L12','U12','th12','L23','U23','th23']
def ls_recover(ts, xs, d=10.0, g=1.0):
    X = np.vstack(xs)
    DX = np.vstack([np.gradient(y, t, axis=0) for t, y in zip(ts, xs)])
    def resid(z):
        p = dict(zip(KEYS, z))
        f1 = -g*X[:,0] + hill_rep(X[:,2],p['L31'],p['U31'],p['th31'],d)
        f2 = -g*X[:,1] + hill_rep(X[:,0],p['L12'],p['U12'],p['th12'],d)
        f3 = -g*X[:,2] + hill_rep(X[:,1],p['L23'],p['U23'],p['th23'],d)
        return np.concatenate([DX[:,0]-f1, DX[:,1]-f2, DX[:,2]-f3])
    z0 = np.array([1.,5.,3., 1.,5.,3., 1.,5.,3.]) * 0.8
    s = least_squares(resid, z0, bounds=(0, 30), max_nfev=20000)
    p = dict(zip(KEYS, s.x)); p['d'] = d; p['g'] = g; return p

p_ls = ls_recover(ts, xs_noisy)
print('LS estimate:', {k: round(p_ls[k],2) for k in KEYS})
print('LS lands in node 13:', in_region(p_ls))

levels = [0.0, 0.1, 0.25, 0.5]; rate = []
for ub in levels:
    r = np.random.default_rng(11); hits = 0
    for _ in range(15):
        xs_n = [add_noise(y, ub, gap, r) for y in xs]
        hits += in_region(ls_recover(ts, xs_n))
    rate.append(hits/15)
print('LS region-recovery rate by noise:', dict(zip(levels, rate)))

**What you should see.** Least squares keeps node 13 at low-to-moderate noise (the rate
stays high), even though its parameter estimates are not perfect - the qualitative regime,
a stable oscillation, survives small errors. At the largest noise it can finally slip, a
reminder that region recovery is a real test, not a formality.

In [ ]:
def build_tensors(ts, xs, x0s):
    # flatten all trajectories into (N,1) time, (N,m) repeated x0, (N,m) measured states
    t_d = np.concatenate([t[:, None] for t in ts])
    x_d = np.vstack(xs)
    x0_d = np.vstack([np.tile(x0, (len(t), 1)) for x0, t in zip(x0s, ts)])
    t_ic = np.zeros((len(x0s), 1)); x_ic = np.array(x0s, dtype=float)  # t=0 anchors
    to = lambda a: torch.tensor(a)
    return (to(t_d), to(x0_d), to(x_d), to(t_ic), to(x_ic))

## 4. The PINN, with Fourier feature mapping

Now the spectral-bias fix. We give the surrogate **Fourier features** in time
(`fourier_k > 0`): instead of feeding the raw time `t`, we feed `sin` and `cos` of several
frequencies, so a periodic solution is easy for the network to represent. The rest of the
PINN is unchanged - learnable `(L, U, theta, d)` from a neutral init, the same composite
loss.

Try setting `fourier_k=0` and re-running: the plain network fits the oscillation poorly and
the recovered parameters degrade, which is the paper's feature-mapping result in miniature.

In [ ]:
# --- the physics-informed network: surrogate u_theta(t, x0) + learnable (L,U,theta,d) ---
class PINN(nn.Module):
    def __init__(self, m, param_init, T, hidden=64, depth=4, fourier_k=0):
        super().__init__()
        self.m, self.T, self.fourier_k = m, T, fourier_k
        in_dim = (2*fourier_k if fourier_k > 0 else 1) + m
        if fourier_k > 0:
            self.register_buffer('freqs', 2*np.pi*torch.arange(1, fourier_k+1).double())
        layers, d0 = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d0, hidden), nn.Tanh()]; d0 = hidden
        layers += [nn.Linear(d0, m)]
        self.net = nn.Sequential(*layers)
        # physical parameters via positive reparametrization  p = raw^2 + eps
        self.raw = nn.ParameterDict(
            {k: nn.Parameter(torch.tensor(float(v)**0.5)) for k, v in param_init.items()})

    def phys_params(self):
        return {k: self.raw[k]**2 + 1e-6 for k in self.raw}

    def _feat(self, t):
        tn = t / self.T   # normalize time to [0,1]; the chain rule is handled by autograd
        if self.fourier_k > 0:
            return torch.cat([torch.sin(self.freqs*tn), torch.cos(self.freqs*tn)], dim=1)
        return tn

    def forward(self, t, x0):
        return self.net(torch.cat([self._feat(t), x0], dim=1))

def fit_pinn(model, data, rhs_t, steps=4000, lr=1e-3, w=(5.0, 1.0, 1.0), log=500):
    t_d, x0_d, x_d, t_ic, x_ic = data   # all tensors
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for it in range(steps):
        opt.zero_grad()
        loss_data = ((model(t_d, x0_d) - x_d)**2).mean()           # fit the measurements
        tc = t_d.clone().requires_grad_(True)                       # physics residual:
        u = model(tc, x0_d)                                         #   du/dt should equal f(u)
        grads = [torch.autograd.grad(u[:, j].sum(), tc, create_graph=True)[0]
                 for j in range(model.m)]
        du = torch.cat(grads, dim=1)
        loss_phys = ((du - rhs_t(u, model.phys_params()))**2).mean()
        loss_ic = ((model(t_ic, x_ic) - x_ic)**2).mean()           # match initial conditions
        loss = w[0]*loss_data + w[1]*loss_phys + w[2]*loss_ic
        loss.backward(); opt.step()
        if log and it % log == 0:
            print(f'  step {it:5d}  data {loss_data.item():.2e}  phys {loss_phys.item():.2e}')
    return model

In [ ]:
def rhs_t(x, p):
    x1, x2, x3 = x[:,0:1], x[:,1:2], x[:,2:3]
    d1 = -p['g']*x1 + hill_rep_t(x3, p['L31'],p['U31'],p['th31'],p['d'])
    d2 = -p['g']*x2 + hill_rep_t(x1, p['L12'],p['U12'],p['th12'],p['d'])
    d3 = -p['g']*x3 + hill_rep_t(x2, p['L23'],p['U23'],p['th23'],p['d'])
    return torch.cat([d1, d2, d3], dim=1)

init = {k: (0.05 if k[0]=='L' else 1.0) for k in KEYS}   # neutral: L~0, U~1, theta~1
init.update(d=5.0, g=1.0)
torch.manual_seed(0)
model = PINN(m=3, param_init=init, T=T, hidden=64, depth=4, fourier_k=5)   # <- features on
data = build_tensors(ts, xs_noisy, x0s)
model = fit_pinn(model, data, rhs_t, steps=8000, lr=1e-3, w=(50.0,10.0,1.0))

pp = {k: float(v) for k, v in model.phys_params().items()}
print('PINN estimate:', {k: round(pp[k],2) for k in KEYS})
print('PINN lands in node 13:', in_region(pp))

---
### The takeaway

Two things to carry away. First, small differences in the parameter estimates need not move
the model out of its DSGRN region, so the qualitative regime - here a sustained oscillation
- is preserved even when the numerical fits differ; that is the reassuring half of the
complementarity story. Second, matching the *kind* of dynamics matters for the method too:
a periodic signal needs Fourier features for the network to represent it, and without them
the recovery suffers.

## Morse graph + Conley-index recovery (DSGRN, optional)

The region test above asks for **exact DSGRN region equality**. A weaker, biologically
natural criterion asks only that the recovered region carry the **same Conley-Morse graph**
as the target - the same recurrent Morse sets, reachability order, and Conley-index labels -
up to label-preserving isomorphism. The implication

$$\text{same region}\ \Rightarrow\ \text{same Morse graph and Conley labels}$$

always holds, so region recovery already *implies* Morse/Conley recovery; the question is
whether the **converse** adds any successes - i.e. whether a miss on the exact region can
still land in a *different* region with an isomorphic Conley-Morse graph.

We use the existing DSGRN tools, not a new pipeline:
`par_index_from_sample` (learned parameters -> region index), `DSGRN.Blowup.ConleyMorseGraph`
(region -> annotated Morse graph), and `DSGRN.isomorphic_morse_graphs` (directed-graph
isomorphism with `node_match` on the **Conley index** - which is location-free, so two fixed
points in different cells match, while a fixed point never matches a periodic orbit).

In [ ]:
# Optional: building DSGRN needs a C++ toolchain (a few minutes on Colab).
# !pip -q install networkx DSGRN
DSGRN_OK = False
try:
    import DSGRN, numpy as np
    NET_SPEC = 'x : ~z\ny : ~x\nz : ~y\n'
    _net = DSGRN.Network(NET_SPEC)
    _pg  = DSGRN.ParameterGraph(_net)
    TARGET = 13
    _mg = {}
    def conley_morse(idx):          # region index -> annotated Conley-Morse graph (cached)
        if idx not in _mg:
            _mg[idx] = DSGRN.Blowup.ConleyMorseGraph(_pg.parameter(idx))[0]
        return _mg[idx]
    def to_matrices(p):             # learned (L, U, theta) -> DSGRN L, U, T matrices
        n = 3
        L = np.zeros((n, n)); U = np.zeros((n, n)); T = np.zeros((n, n))
        L[2,0],U[2,0]=p['L31'],p['U31']; L[0,1],U[0,1]=p['L12'],p['U12']; L[1,2],U[1,2]=p['L23'],p['U23']
        T[2,0],T[0,1],T[1,2]=p['th31'],p['th12'],p['th23']
        return L, U, T
    def region_of(p):               # learned parameters -> DSGRN region index (-1 if outside)
        L, U, T = to_matrices(p)
        return DSGRN.par_index_from_sample(_pg, L, U, T)
    def morse_recovers(idx, target=TARGET):   # same Conley-Morse graph up to label-preserving iso
        return idx >= 0 and DSGRN.isomorphic_morse_graphs(conley_morse(idx), conley_morse(target))
    print('target region', TARGET, 'Conley indices:',
          [conley_morse(TARGET).vertex_label(v)[2] for v in conley_morse(TARGET).vertices()])
    DSGRN_OK = True
except Exception as e:
    print('DSGRN unavailable - skipping this section:', repr(e)[:90])

In [ ]:
if DSGRN_OK:
    # the recovered parameters from the cells above
    p_ls = ls_recover(ts, xs_noisy)
    print('recovered region index:', region_of(p_ls),
          '| exact region match:', region_of(p_ls) == TARGET,
          '| Morse/Conley match:', morse_recovers(region_of(p_ls)))

    # noise sweep: exact-region recovery vs Morse/Conley recovery (least squares)
    levels = [0.0, 0.1, 0.25, 0.5]
    print(f'{"noise":>6} | {"exact region":>12} | {"Morse/Conley":>12} | {"region-miss, Morse-ok":>20}')
    for ub in levels:
        r = np.random.default_rng(7); reg = mor = gapc = 0
        for _ in range(15):
            xs_n = [add_noise(y, ub, gap, r) for y in xs]
            idx = region_of(ls_recover(ts, xs_n))
            er = (idx == TARGET); mr = morse_recovers(idx)
            reg += er; mor += mr; gapc += (mr and not er)
        print(f'{ub:>6} | {reg:>10}/15 | {mor:>10}/15 | {gapc:>18}/15')
    # repressilator: node 13 is the unique region with a periodic orbit (FC)

**What you should see.** Among the 27 regions of this network, node 13 is the **only** one
whose Conley-Morse graph contains a periodic orbit (the `FC` Morse set); the other 26 are
monostable. A periodic orbit cannot be matched by a fixed point, so the only way to recover
the Morse graph is to recover node 13 itself: the `region-miss, Morse-ok` column is `0/15`
throughout, and Morse/Conley recovery equals exact region recovery. For oscillation, the
qualitative criterion buys nothing the exact one did not already give.